In [1]:
import gzip
import json
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import glob

In [2]:
# Memory-efficient data loader: processes large JSON files in chunks to prevent RAM overflow
def load_efficiently(path, columns_to_keep):
    """Loads JSON in chunks and keeps only specific columns to save RAM."""
    chunks = []
    # Read in chunks of 100,000 lines at a time
    reader = pd.read_json(path, lines=True, compression='gzip', chunksize=100000)
    
    for chunk in reader:
        # Keep only the columns necessary for your volatility analysis
        # This prevents RAM from ballooning
        chunks.append(chunk[columns_to_keep])
        
    return pd.concat(chunks, ignore_index=True)

STATE_NAME = "Vermont"
meta_path = f"data/meta-{STATE_NAME}.json.gz"
review_path = f"data/review-{STATE_NAME}.json.gz"

# Define only the columns your analysis actually uses
meta_cols = [
    'gmap_id', 
    'name', 
    'category', 
    'avg_rating', 
    'num_of_reviews', 
    'latitude',
    'longitude']
review_cols = ['gmap_id', 'rating', 'text']

df_meta = load_efficiently(meta_path, meta_cols)
df_review = load_efficiently(review_path, review_cols)

Keep only business id, avg rating, and total # of reviews. Also drops rows with missing values

In [3]:
biz = df_meta[["gmap_id", "avg_rating", "num_of_reviews"]].dropna()

biz["num_of_reviews"] = biz["num_of_reviews"].astype(int)
biz["avg_rating"] = biz["avg_rating"].astype(float)


In [4]:
# Binning: Grouping businesses by review volume to analyze statistical variance
bins = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, biz["num_of_reviews"].max() + 1]
labels = [f"{bins[i]}–{bins[i+1]-1}" for i in range(len(bins)-1)]

biz["review_bin"] = pd.cut(biz["num_of_reviews"], bins=bins, labels=labels, right=False)

#Calculate statistics for each bin
stability_table = (biz.groupby("review_bin").agg(
        n_businesses=("gmap_id", "count"),
        rating_std=("avg_rating", "std"),
        rating_var=("avg_rating", "var"),
        mean_rating=("avg_rating", "mean"),
    ).reset_index())

print(stability_table)

  review_bin  n_businesses  rating_std  rating_var  mean_rating
0        1–1           597    1.077194    1.160346     4.397487
1        2–4          1407    0.791099    0.625837     4.391045
2        5–9          2240    0.641285    0.411247     4.385848
3      10–19          1623    0.535512    0.286773     4.419285
4      20–49          1966    0.483702    0.233967     4.407630
5      50–99          1387    0.434095    0.188438     4.424081
6    100–199           997    0.403046    0.162446     4.367803
7    200–499           809    0.382720    0.146474     4.316440
8    500–999           199    0.393118    0.154541     4.302010
9  1000–5498            66    0.357614    0.127888     4.363636


C:\Users\aiden\AppData\Local\Temp\ipykernel_34636\2459605819.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stability_table = (biz.groupby("review_bin").agg(


In [5]:
#Change in std between bins
stability_table["delta_std"] = stability_table["rating_std"].diff().abs()
stability_table

epsilon = 0.02  # ratings barely change beyond this
stable_bins = stability_table[stability_table["delta_std"] < epsilon]
print(stable_bins)


  review_bin  n_businesses  rating_std  rating_var  mean_rating  delta_std
8    500–999           199    0.393118    0.154541      4.30201   0.010398


In [6]:
#40+ reviews is stable (good marker?)
biz_stable = biz[biz["num_of_reviews"] >= 40]

In [7]:
biz["weight"] = np.log1p(biz["num_of_reviews"])

In [8]:
print(df_meta.columns)

Index(['gmap_id', 'name', 'category', 'avg_rating', 'num_of_reviews',
       'latitude', 'longitude'],
      dtype='object')


In [9]:
def add_geo_cells(df, lat_col="latitude", lon_col="longitude", cell_size=0.1):
    d = df.copy()
    d["lat_cell"] = (d[lat_col] // cell_size) * cell_size
    d["lon_cell"] = (d[lon_col] // cell_size) * cell_size
    d["geo_cell"] = d["lat_cell"].astype(str) + "_" + d["lon_cell"].astype(str)
    return d

df_meta_cells = add_geo_cells(df_meta)


In [10]:
#Calculate statistics for each "grid"
cell_features = (
    df_meta_cells.groupby("geo_cell").agg(
        n_businesses=("gmap_id", "nunique"),
        avg_reviews_per_biz=("num_of_reviews", "mean"),
        median_reviews_per_biz=("num_of_reviews", "median"),
        pct_high_review_biz=("num_of_reviews", lambda x: (x >= 100).mean()),
        avg_rating=("avg_rating", "mean"),
    ).reset_index())


In [11]:
if "categories" in df_meta_cells.columns:
    cat_div = (
        df_meta_cells.explode("categories")
        .groupby("geo_cell")["categories"]
        .nunique()
        .rename("category_diversity")
    )
    cell_features = cell_features.merge(cat_div, on="geo_cell", how="left")

In [12]:
# Urbanization Index: Standardizes business density metrics to classify regions (Rural/Suburban/Urban)
features = ["n_businesses", "avg_reviews_per_biz", "pct_high_review_biz",]

if "category_diversity" in cell_features.columns:
    features.append("category_diversity")

X = cell_features[features].fillna(0)

#z-scores Ex: n_businesses = 500 but pct_high_review_biz = 0.25
scaler = StandardScaler()
#turn raw nums into standardized z-scores and get mean of all scores of each cell
cell_features["urban_index"] = scaler.fit_transform(X).mean(axis=1)

In [13]:
# Split into 3 equal-sized groups: Rural, Suburban, Urban
cell_features["urban_rural"] = pd.qcut(cell_features["urban_index"], q=3, labels=["rural", "suburban", "urban"])

In [14]:
#give each biz label
df_meta_cells = df_meta_cells.merge(cell_features[["geo_cell", "urban_rural", "urban_index"]], on="geo_cell", how="left")

#same thing
df_review = df_review.merge(df_meta_cells[["gmap_id", "urban_rural", "urban_index"]], on="gmap_id", how="left")


In [15]:
df_meta_cells.groupby("urban_rural").agg(
    avg_rating=("avg_rating", "mean"),
    median_reviews=("num_of_reviews", "median"),
    pct_5star=("avg_rating", lambda x: (x >= 4.5).mean()), #review with Dhruv or Ansh
)


C:\Users\aiden\AppData\Local\Temp\ipykernel_34636\3960716687.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_meta_cells.groupby("urban_rural").agg(


,avg_rating,median_reviews,pct_5star
urban_rural,,,
rural,4.569384,7.0,0.748752
suburban,4.486596,14.0,0.665184
urban,4.361179,26.0,0.541498


In [16]:
df_review["review_len"] = df_review["text"].fillna("").str.len()
df_review.drop(columns=["text"], inplace=True)

review_summary = df_review.groupby("urban_rural").agg(
    avg_review_rating=("rating", "mean"),
    avg_review_length=("review_len", "mean"),
    n_reviews=("gmap_id", "size"),
)

# 4. Reorder for a logical flow: Rural -> Suburban -> Urban
review_summary = review_summary.reindex(["rural", "suburban", "urban"])

print(review_summary)

             avg_review_rating  avg_review_length  n_reviews
urban_rural                                                 
rural                 4.590123         105.392049       7647
suburban              4.497063          88.085062      69655
urban                 4.338099          96.328231     776247


C:\Users\aiden\AppData\Local\Temp\ipykernel_34636\1311528783.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  review_summary = df_review.groupby("urban_rural").agg(


In [17]:
#groups bizs into bins and calculates density metrics
def extract_cell_features(df):
    d = df.copy()
    
    #creates grid
    CELL_SIZE = 0.1 
    d["lat_cell"] = (d["latitude"] // CELL_SIZE) * CELL_SIZE
    d["lon_cell"] = (d["longitude"] // CELL_SIZE) * CELL_SIZE
    d["geo_cell"] = d["lat_cell"].astype(str) + "_" + d["lon_cell"].astype(str)
    
    #calculate features per cell
    features = d.groupby("geo_cell").agg(
        n_businesses=("gmap_id", "nunique"),
        avg_reviews_per_biz=("num_of_reviews", "mean"),
        pct_high_review_biz=("num_of_reviews", lambda x: (x >= 100).mean())
    )
    
    return features, d

In [18]:
# finds every file in data
all_files = glob.glob("data/meta-*.json.gz")
global_feature_list = []

# Define the columns needed for extract_cell_features
needed_cols = ["gmap_id", "num_of_reviews", "latitude", "longitude"]

for file in all_files:
    # Use the correct function name: load_efficiently
    temp_df = load_efficiently(file, needed_cols).dropna()
    feats, _ = extract_cell_features(temp_df)
    global_feature_list.append(feats)

# combine cells
combined_cells = pd.concat(global_feature_list)
scaler = StandardScaler().fit(combined_cells)

# Calculate thresholds
global_index = scaler.transform(combined_cells).mean(axis=1)
low_threshold = np.percentile(global_index, 33.3)  # Rural ends here
high_threshold = np.percentile(global_index, 66.6) # Suburban ends here

In [19]:
feats, df_with_cells = extract_cell_features(df_meta)

#scale state using global scalar
# Wrap state_scores in a pd.Series
state_scores = pd.Series(scaler.transform(feats).mean(axis=1), index=feats.index)


#apply label based on threshold
def classify_region(val):
    if val < low_threshold: return "rural"
    if val < high_threshold: return "suburban"
    return "urban"

feats["region_type"] = state_scores.apply(classify_region)

#merge back to the metadata
df_meta = df_with_cells.merge(feats[["region_type"]], on="geo_cell", how="left")

In [20]:
# Binning: Grouping businesses by review volume to analyze statistical variance
bins = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, df_meta["num_of_reviews"].max() + 1]
labels = [f"{bins[i]}–{bins[i+1]-1}" for i in range(len(bins)-1)]

df_meta["review_bin"] = pd.cut(df_meta["num_of_reviews"], bins=bins, labels=labels, right=False)

regional_summary = df_meta.groupby(["region_type", "review_bin"]).agg(
    rating_std=("avg_rating", "std"), 
    n_businesses=("gmap_id", "count")
).reset_index()

C:\Users\aiden\AppData\Local\Temp\ipykernel_34636\1770428031.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  regional_summary = df_meta.groupby(["region_type", "review_bin"]).agg(


In [21]:
#grouping for tableau
regional_summary = df_meta.groupby(["region_type", "review_bin"]).agg(
    rating_std=("avg_rating", "std"), 
    n_businesses=("gmap_id", "count")
).reset_index()

C:\Users\aiden\AppData\Local\Temp\ipykernel_34636\555981880.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  regional_summary = df_meta.groupby(["region_type", "review_bin"]).agg(


In [22]:
print(all_files)

['data\\meta-California.json.gz', 'data\\meta-District_of_Columbia.json.gz', 'data\\meta-Florida.json.gz', 'data\\meta-New_York.json.gz', 'data\\meta-Texas.json.gz', 'data\\meta-Vermont.json.gz', 'data\\meta-Virginia.json.gz', 'data\\meta-Washington.json.gz', 'data\\meta-Wyoming.json.gz']


In [23]:
regional_summary

,region_type,review_bin,rating_std,n_businesses
0,rural,1–1,1.027569,72
1,rural,2–4,0.770897,113
2,rural,5–9,0.474704,155
3,rural,10–19,0.435324,67
4,rural,20–49,0.532450,49
5,rural,50–99,0.235884,13
6,rural,100–199,NaN,0
7,rural,200–499,NaN,0
8,rural,500–999,NaN,0
9,rural,1000–5498,NaN,0


In [24]:
%pip install plotly nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
# Visualization: Plotting rating trends to identify stability thresholds
import plotly.express as px

#Drop blank biz values
plot_data = regional_summary.dropna(subset=['rating_std'])

fig = px.line(
    plot_data, 
    x="review_bin", 
    y="rating_std", 
    color="region_type",
    markers=True,
    title="Rating Stability (Std Dev) by Review Volume and Region",
    labels={"rating_std": "Rating Standard Deviation", "review_bin": "Number of Reviews"},
    category_orders={"review_bin": labels} # Ensures bins stay in numerical order
)

fig.update_layout(template="plotly_white", hovermode="x unified")
fig.show()

In [26]:
meta_summary = df_meta.groupby("region_type").agg(
    avg_rating=("avg_rating", "mean"),
    median_reviews=("num_of_reviews", "median"),
    pct_high_rated=("avg_rating", lambda x: (x >= 4.5).mean()),
    total_businesses=("gmap_id", "count")
).reindex(["rural", "suburban", "urban"]) # Keeps them in logical order

print("--- Metadata Summary (Business Level) ---")
print(meta_summary)

--- Metadata Summary (Business Level) ---
             avg_rating  median_reviews  pct_high_rated  total_businesses
region_type                                                              
rural          4.539019             6.0        0.729211               469
suburban       4.452596            17.0        0.628767              3949
urban          4.347534            28.0        0.529027              6873


In [27]:
df_review = df_review.merge(
    df_meta[['gmap_id', 'region_type']], 
    on='gmap_id', 
    how='left'
)

# calculate length BEFORE dropping the text column
if 'text' in df_review.columns:
    df_review["review_len"] = df_review["text"].fillna("").str.len()
    df_review.drop(columns=["text"], inplace=True)
else:
    print("Error: 'text' column missing. Please re-run your data loading cell.")

# Generate summary
review_summary = df_review.groupby("region_type").agg(
    avg_review_rating=("rating", "mean"),
    avg_review_length=("review_len", "mean"),
    n_reviews=("gmap_id", "size"),
)

print(review_summary)

Error: 'text' column missing. Please re-run your data loading cell.
             avg_review_rating  avg_review_length  n_reviews
region_type                                                 
rural                 4.581413         106.859797       5221
suburban              4.429900          85.415747     184877
urban                 4.331036          98.958752     666143


In [28]:
# --- EXPORT REGIONAL SUMMARIES FOR TABLEAU ---

# Add the state column to the summaries so you can compare states in Tableau
meta_summary['state'] = STATE_NAME
review_summary['state'] = STATE_NAME

# Use the STATE_NAME variable to create unique filenames
meta_summary.to_csv(f'{STATE_NAME}_regional_business_summary.csv', index=True)
review_summary.to_csv(f'{STATE_NAME}_regional_user_behavior.csv', index=True)

print(f"Exported summaries for {STATE_NAME}")

Exported summaries for Vermont
